The code in this notebook was adapted from assignments I did for the Fundamentals of Natural Language Processing course by University of Colorado Boulder.

A function that turns a tokenized corpus List[List[int]] into trigrams, i.e. tuples of 3 integers List[Tuple[int, int, int]].


In [ ]:
def get_trigrams(corpus: List[List[int]]) -> List[Tuple[int, int, int]]:
    """Return a list of trigrams in `corpus`."""
    
    trigram = ()
    trigrams = []
    for sentence in corpus:
        for word in range(len(sentence) - 2):
            trigram = (sentence[word], sentence[word + 1], sentence[word + 2])
            trigrams.append(trigram)
  
    return trigrams

A function that computes the conditional probability for trigrams.

In [ ]:
def conditional_prob_trigram(w_3, w_2, w_1, corpus: List[List[int]]) -> float:
    """Return the conditional probability of `w_3` given `w_2` and `w_1` in `corpus`.
    P(w_3 | w_2, w_1)_Laplace = count(w_1, w_2, w_3) + 1 / count(w_1, w_2) + V
    Uses Laplace (add-one) smoothing.
    """
    tokens = [token for tokenized_sentence in corpus for token in tokenized_sentence]
    bigrams = get_bigrams(corpus)
    trigrams = get_trigrams(corpus)
    
    bigram_counts = Counter(bigrams)
    trigram_counts = Counter(trigrams)

    vocab = set(tokens)
    V = len(vocab)
    probability = (trigram_counts[(w_1, w_2, w_3)] + 1)/(bigram_counts[(w_1, w_2)] + V)
    
    return probability

A function that predicts the next word in a trigram language model.

In [ ]:
def predict_next_word_trigram(w_2, w_1, corpus: List[List[int]]) -> int:
    """Return the most likely next word given `w_2` and `w_1` in `corpus`."""

    probs = {}
    for sentence in corpus:
        for token in sentence:
            cprob = conditional_prob_trigram(token, w_2, w_1, corpus)
            probs[token] = cprob
    most_likely_next_word = max(probs, key=probs.get)
   
    return most_likely_next_word 

A function that predicts the next sentence in a trigram language model.

In [ ]:
stop_token_id = token_to_id['</s>']
def predict_sentence_trigrams(initial_sequence: List[int], corpus: List[List[int]], limit: int = 5) -> List[int]:
    """Return a sentence of `len(initial_sequence)`+`limit` words 
    predicted from `initial_sequence` in `corpus`."""

    predicted_sequence = list(initial_sequence)
    count = 0
    while count<=limit:
        initial_string = predicted_sequence[-2:]
        predicted_word = predict_next_word_trigram(initial_string[1], initial_string[0], corpus)
        if predicted_word == stop_token_id:
            break
        predicted_sequence.append(predicted_word)
        count+=1
       
    return predicted_sequence